In [50]:
# 1️⃣ 실행 환경 감지
try:
    import google.colab
    IS_COLAB = True
    print("✅ Google Colab")
except ImportError:
    IS_COLAB = False
    print("✅ 로컬 Jupyter")


✅ 로컬 Jupyter


In [51]:
# 3️⃣ API 키 로드
import os

if IS_COLAB:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
else:
    from dotenv import load_dotenv
    load_dotenv()

print("✅ API 키 OK" if os.environ.get("OPENAI_API_KEY") else "❌ 키 없음")


✅ API 키 OK


In [52]:
from typing import Literal, Optional
from pydantic import BaseModel, Field   # 1-2교시 정형화 도구
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langchain_core.tools import tool   # ⭐ 오늘의 주인공

# Tool Calling은 temperature=0 권장 (도구 선택의 일관성)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

print("✅ import 완료")

✅ import 완료


In [54]:
# 헬퍼 함수: 질문 + 도구 → 4단계 흐름 실행 → 최종 답변
def run_with_tools(question: str, tools: list, llm_obj=llm):
    llm_with_tools = llm_obj.bind_tools(tools)
    messages: list[HumanMessage | AIMessage | ToolMessage] = [HumanMessage(content=question)]

    response = llm_with_tools.invoke(messages)
    messages.append(response)

    if not response.tool_calls:
        return response.content

    tool_map = {t.name: t for t in tools}
    for tc in response.tool_calls:
        result = tool_map[tc["name"]].invoke(tc["args"])
        messages.append(ToolMessage(content=str(result), tool_call_id=tc["id"]))

    return llm_with_tools.invoke(messages).content

print("✅ 헬퍼 함수 준비 완료")

✅ 헬퍼 함수 준비 완료


테스트: "180cm가 몇 미터야?", "10km는 몇 마일이야?"

In [55]:
# Pydantic 스키마로 변환 결과의 필드와 타입을 일정하게 유지합니다.
class ConversionResult(BaseModel):
    original_value: float = Field(description="변환 전 값")
    original_unit: str = Field(description="변환 전 단위")
    converted_value: float = Field(description="변환 후 값")
    converted_unit: str = Field(description="변환 후 단위")

LENGTH_TO_M = {"m": 1.0, "cm": 0.01, "km": 1000.0, "inch": 0.0254, "ft": 0.3048, "mile": 1609.344}
LengthUnit = Literal["m", "cm", "km", "inch", "ft", "mile"]

@tool
def convert_length(value: float, from_unit: LengthUnit, to_unit: LengthUnit) -> ConversionResult:
    """길이 단위를 변환합니다.

    Args:
        value: 변환할 값
        from_unit: 출발 단위 ('m', 'cm', 'km', 'inch', 'ft', 'mile')
        to_unit: 도착 단위 ('m', 'cm', 'km', 'inch', 'ft', 'mile')
    """
    converted = value * LENGTH_TO_M[from_unit] / LENGTH_TO_M[to_unit] # 1m -> cm: 1 * 1 / 0.01 = 100cm
    return ConversionResult(original_value=value, original_unit=from_unit,
                            converted_value=converted, converted_unit=to_unit)


In [57]:
QUERIES = [
    "180cm가 몇 미터야?",
    "10km는 몇 마일이야?",
    "5ft가 몇 cm야?",
    "100inch를 km로 바꿔줘.",
    "13mile을 m로 변환해줘.",
]

for q in QUERIES:
    print(f"❓ {q}")
    print(f"💬 {run_with_tools(q, [convert_length])}")
    print()

❓ 180cm가 몇 미터야?
💬 180cm는 1.8미터입니다.

❓ 10km는 몇 마일이야?
💬 10킬로미터는 약 6.21마일입니다.

❓ 5ft가 몇 cm야?
💬 5피트는 152.4센티미터입니다.

❓ 100inch를 km로 바꿔줘.
💬 100 인치는 0.00254 킬로미터입니다.

❓ 13mile을 m로 변환해줘.
💬 13 마일은 약 20,921.472 미터입니다.



In [59]:
from datetime import date

# 날짜 원문과 계산 결과를 일정한 구조로 반환하기 위한 스키마입니다.
class DateDiff(BaseModel):
    date1: str = Field(description="시작 날짜")
    date2: str = Field(description="종료 날짜")
    days: int = Field(description="두 날짜 사이의 전체 일수")
    weeks: int = Field(description="전체 일수를 기준으로 완전히 지난 주 수")

# 이 함수를 LangChain이 호출할 수 있는 도구로 등록합니다.
@tool
def days_between(date1: str, date2: str) -> DateDiff:
    """두 날짜(YYYY-MM-DD) 사이의 일수·주수를 반환합니다."""
    # ISO 날짜 문자열을 파싱해 빼고, 입력 순서와 무관하게 양수 일수로 만듭니다.
    days = abs((date.fromisoformat(date2) - date.fromisoformat(date1)).days)
    # 7로 나눈 몫은 남는 일수를 제외한 완전한 주 수입니다.
    return DateDiff(date1=date1, date2=date2, days=days, weeks=days // 7)

In [62]:
QUERIES = [
    "26년 10월 25일 부터 12월 25일까지 며칠이야?",
    "20261001 부터 3주 뒤는 몇일이야?",
    "2022-01-01 부터 2022-01-29까지 며칠이야?"
    "2021-02-01 부터 2021-02-28까지 몇주, 몇일이야?"
]

for q in QUERIES:
    print(f"❓ {q}")
    print(f"💬 {run_with_tools(q, [days_between])}")
    print()

❓ 26년 10월 25일 부터 12월 25일까지 며칠이야?
💬 2026년 10월 25일부터 12월 25일까지는 61일입니다. 이는 8주에 해당합니다.

❓ 20261001 부터 3주 뒤는 몇일이야?
💬 2026년 10월 1일부터 3주 뒤는 2026년 10월 22일입니다.

❓ 2022-01-01 부터 2022-01-29까지 며칠이야?2021-02-01 부터 2021-02-28까지 몇주, 몇일이야?
💬 - 2022-01-01 부터 2022-01-29까지는 **28일**입니다.
- 2021-02-01 부터 2021-02-28까지는 **3주 6일**입니다.



테스트: "서울이랑 부산 날씨 알려줘"   ← 도구를 두 번 호출!

In [72]:
# 날씨 도구가 반환할 필드와 타입을 정의하는 Pydantic 스키마입니다.
class WeatherInfo(BaseModel):
    city: str = Field(description="조회한 도시 이름")
    temperature_c: float = Field(description="섭씨 온도")
    condition: Literal["맑음", "흐림", "비", "눈"] = Field(description="날씨 상태")
    humidity_percent: int = Field(description="습도")

weather = {
        "서울": (18.0, "맑음", 45),
        "부산": (23.0, "흐림", 60),
        "대구": (21.0, "비", 80),
        "대관령": (10.0, "눈", 90)
    }

# 도시를 입력받는 함수를 LangChain 도구로 등록합니다.
@tool
def get_weather(city: str) -> WeatherInfo:
    """도시의 데모 날씨 정보를 조회합니다."""

    # 튜플 값을 의미가 드러나는 변수명으로 나눠 사용합니다.
    temperature, condition, humidity = weather[city]
    # 스키마 객체로 반환해 필드 이름과 데이터 타입을 명확히 합니다.
    return WeatherInfo(
        city=city,  # 요청한 도시 이름을 포함합니다.
        temperature_c=temperature,  # 데모 섭씨 온도를 넣습니다.
        condition=condition,  # 데모 날씨 상태를 넣습니다.
        humidity_percent=humidity,  # 데모 습도를 넣습니다.
    )


In [73]:
QUERIES = [
    "부산 날씨는 어때?",
    "서울의 날씨와 기온은?",
    "대구 날씨 알려줘",
    "대관령에 눈이 오고 있니?"
]

for q in QUERIES:
    print(f"❓ {q}")
    print(f"💬 {run_with_tools(q, [get_weather])}")
    print()

❓ 부산 날씨는 어때?
💬 부산의 현재 날씨는 흐림이며, 기온은 23도, 습도는 60%입니다.

❓ 서울의 날씨와 기온은?
💬 서울의 날씨는 맑고, 기온은 18도이며, 습도는 45%입니다.

❓ 대구 날씨 알려줘
💬 대구의 현재 날씨는 다음과 같습니다:
- 기온: 21도
- 날씨: 비
- 습도: 80%

❓ 대관령에 눈이 오고 있니?
💬 대관령에 현재 눈이 오고 있으며, 기온은 10도, 습도는 90%입니다.



In [71]:
@tool
def add_v1(a: int, b: int) -> int:
    """더하기"""        # ← 너무 짧음
    return a + b

@tool
def add_v2(a: int, b: int) -> int:
    """두 정수의 합을 계산합니다.
    수학 계산이 필요한 모든 경우에 사용하세요."""
    return a + b